In [1]:
from chunk_qa_panel import render_qa_result
from cmap_qa_panel import render_cmap_qa_result

from llms_kgs.logic.cmap_qa import CMapQAResult
from llms_kgs.logic.chunk_qa import ChunkQAResult

import json

with open("../data/rag/gemma_chunk_rag.json", "r") as f:
    gemma_chunk_results = [ChunkQAResult.from_dict(x) for x in json.load(f)]

with open("../data/rag/gemini_chunk_rag.json", "r") as f:
    gemini_chunk_results = [ChunkQAResult.from_dict(x) for x in json.load(f)]

with open("../data/rag/gemma_cmap_rag.json", "r") as f:
    gemma_cmap_results = [CMapQAResult.from_dict(x) for x in json.load(f)]

with open("../data/rag/gemini_cmap_rag.json", "r") as f:
    gemini_cmap_results = [CMapQAResult.from_dict(x) for x in json.load(f)]

Chunk Rag of Gemini for question: "What are the main symptoms of depression?

In [4]:
render_qa_result(gemini_chunk_results[0])

Chunk Rag of Gemma for question: "What are the main symptoms of depression?

In [3]:
render_qa_result(gemma_chunk_results[4])

Cmap Rag of Gemma for question: "What are the main symptoms of depression?

In [4]:
render_cmap_qa_result(gemma_cmap_results[4])

CMap Rag of Gemini for question: "What are the main symptoms of depression?

In [5]:
render_cmap_qa_result(gemini_cmap_results[4])

Source,Relation,Target
Not enough information is given to answer the question,→ @ →,


In [6]:
def recover_source_map(answer_triples, retrieved_cmaps):
    """
    Matches answer triples back to their source concept map.
    """
    citation_map = {}
    
    for triple in answer_triples:
        # Create a string signature for matching
        triple_str = f"{triple.source} @ {triple.relation} @ {triple.target}"
        
        found = False
        for i, cmap in enumerate(retrieved_cmaps):
            # Check if this triple exists in this map's triples
            # We check strictly string equality here
            for cmap_triple in cmap.triples:
                if (cmap_triple.source.label == triple.source and 
                    cmap_triple.relation.label == triple.relation and 
                    cmap_triple.target.label == triple.target):
                    
                    citation_map[triple_str] = f"Map {i}" # e.g., "Map 0"
                    found = True
                    break
            if found: break
            
        if not found:
            citation_map[triple_str] = "Inferred/External"

    return citation_map

# --- HOW TO USE IT ---
# 1. Get the result object you already computed
gemma_result = gemma_cmap_results[3] 

# 2. Run the recovery
citations = recover_source_map(
    gemma_result.answer.triples, 
    gemma_result.retrieved_cmaps
)

# 3. Print for LaTeX
print("Evidencia (Ternas):")
for t in gemma_result.answer.triples:
    t_str = f"{t.source} @ {t.relation} @ {t.target}"
    source_id = citations.get(t_str, "?")
    # Output format: \item [Map 0] (Source, Rel, Target)
    print(f"\\item \\textbf{[{source_id}]} ({t.source}, {t.relation}, {t.target})")

Evidencia (Ternas):
\item \textbf[{'Map 0'}] (introversion, involves orientation away from, outer world of people and things)
\item \textbf[{'Map 1'}] (extraversion, is characterized by, orientation of interests and energies toward the outer world of people and things)
\item \textbf[{'Map 0'}] (introverts, are, relatively more withdrawn)
\item \textbf[{'Map 1'}] (extraverts, are, outgoing)
\item \textbf[{'Map 0'}] (introverts, are, relatively more retiring)
\item \textbf[{'Map 1'}] (extraverts, are, gregarious)
\item \textbf[{'Map 0'}] (introverts, are, relatively more reserved)
\item \textbf[{'Map 1'}] (extraverts, are, sociable)
\item \textbf[{'Map 0'}] (introverts, are, relatively more quiet)
\item \textbf[{'Map 1'}] (extraverts, are, openly expressive)


In [10]:
print(gemma_cmap_results[1].extractor_invocation.raw_answer)

{"triples": [
  {
    "source": "cerebrum",
    "relation": "comprises",
    "target": "cerebral hemispheres"
  },
  {
    "source": "cerebral hemispheres",
    "relation": "include",
    "target": "basal ganglia"
  },
  {
    "source": "cerebral hemispheres",
    "relation": "include",
    "target": "amygdala"
  },
  {
    "source": "cerebral hemispheres",
    "relation": "include",
    "target": "hippocampus"
  }
], "final_answer": "The brain has cerebral hemispheres."}


In [5]:
import re
import textwrap

def generate_chunk_latex_atomic(retrieved_docs, caption, label):
    """
    Genera una longtable robusta para Chunks de texto.
    Divide el texto en párrafos/bloques para asegurar que LaTeX pueda 
    romper la página (page break) dentro de un mismo chunk si es muy largo.
    """
    
    def escape_latex(text):
        if not isinstance(text, str): text = str(text)
        replacements = {'&': '\\&', '%': '\\%', '$': '\\$', '#': '\\#', '_': '\\_', '{': '\\{', '}': '\\}', '~': '\\textasciitilde{}', '^': '\\textasciicircum{}', '\\': '\\textbackslash{}'}
        for k, v in replacements.items(): text = text.replace(k, v)
        return text

    # --- HEADER ---
    print(f"% --- Table for: {label} ---")
    # Usamos la misma estructura que la tabla de mapas para consistencia visual
    # Requiere \usepackage{array} para >{\raggedright\arraybackslash}
    print("\\begin{longtable}{|p{1.5cm}|>{\\raggedright\\arraybackslash}p{13cm}|}") 
    print(f"\\caption{{{caption}}} \\label{{{label}}} \\\\")
    print("\\hline")
    print("\\textbf{Ranking} & \\textbf{Contenido del Documento} \\\\")
    print("\\hline")
    print("\\endfirsthead")
    
    print("\\hline")
    print("\\textbf{Ranking} & \\textbf{Contenido (cont.)} \\\\")
    print("\\hline")
    print("\\endhead")

    # --- BODY ---
    for i, doc in enumerate(retrieved_docs):
        # 1. Obtener el texto crudo
        # Soporta objetos Document (LangChain) o strings puros
        raw_text = doc.page_content if hasattr(doc, 'page_content') else str(doc)
        
        # 2. Limpieza y Estrategia de División (Chunking the Chunk)
        # Dividimos por saltos de línea dobles (párrafos) o simples
        paragraphs = re.split(r'\n+', raw_text.strip())
        
        # FILTRO DE SEGURIDAD:
        # Si un párrafo es monstruosamente largo (ej. >1000 chars sin enters),
        # longtable podría fallar. Lo forzamos a dividirse visualmente.
        safe_paragraphs = []
        for p in paragraphs:
            if len(p) > 800: # Si es gigante, lo cortamos en trozos de 800 chars
                safe_paragraphs.extend(textwrap.wrap(p, width=800)) # Esto devuelve lista
            else:
                safe_paragraphs.append(p)

        # 3. Imprimir Filas
        for j, para in enumerate(safe_paragraphs):
            clean_text = escape_latex(para)
            
            # Lógica de Columnas:
            # - Si es el primer párrafo (j=0), imprimimos el Ranking 'i'.
            # - Si son párrafos siguientes, la columna de ranking va vacía.
            rank_col = str(i) if j == 0 else ""
            
            # Imprimir fila atómica
            if clean_text.strip(): # Solo si hay texto
                print(f"{rank_col} & {clean_text} \\\\")
        
        # SEPARADOR: Línea horizontal solo al terminar el chunk completo
        print("\\hline")

    print("\\end{longtable}")
    print("\n" * 2)

In [16]:
generate_chunk_latex_atomic(
    [chunk.text for chunk in gemini_chunk_results[3].retrieved_chunks],
    "Contexto recuperado (Chunks) para Caso 4: Introversión y Extroversión",
    "tab:context_c4_naive")

% --- Table for: tab:context_c4_naive ---
\begin{longtable}{|p{1.5cm}|>{\raggedright\arraybackslash}p{13cm}|}
\caption{Contexto recuperado (Chunks) para Caso 4: Introversión y Extroversión} \label{tab:context_c4_naive} \\
\hline
\textbf{Ranking} & \textbf{Contenido del Documento} \\
\hline
\endfirsthead
\hline
\textbf{Ranking} & \textbf{Contenido (cont.)} \\
\hline
\endhead
0 & Introversion is an orientation toward the internal private world of one’s self and one’s inner thoughts and feelings, rather than toward the outer world of people and things. \\
 & Introversion is a broad personality trait and, like extraversion, exists on a continuum of attitudes and behaviors. \\
 & Introverts are relatively more withdrawn, retiring, reserved, quiet, and deliberate; they may tend to mute or guard expression of positive affect, adopt more skeptical views or positions, and prefer to work independently. \\
\hline
1 & Extraversion is one of the elements of the Big Five and five-factor personality 

In [8]:
import re

def generate_cmap_latex_atomic(retrieved_maps, caption, label):
    """
    Genera una longtable donde CADA ELEMENTO (Pregunta, Terna) es una fila independiente.
    Esto permite que la tabla se corte en cualquier punto entre páginas.
    """
    
    def escape_latex(text):
        if not isinstance(text, str): text = str(text)
        replacements = {'&': '\\&', '%': '\\%', '$': '\\$', '#': '\\#', '_': '\\_', '{': '\\{', '}': '\\}', '~': '\\textasciitilde{}', '^': '\\textasciicircum{}', '\\': '\\textbackslash{}'}
        for k, v in replacements.items(): text = text.replace(k, v)
        return text

    def format_triple_content(t):
        s_str, r_str, o_str = "", "", ""
        if isinstance(t, dict):
            s_str = t.get('source', {}).get('label', '')
            r_str = t.get('relation', {}).get('label', '')
            o_str = t.get('target', {}).get('label', '')
        else:
            if hasattr(t, 'source'): s_str = t.source.label if hasattr(t.source, 'label') else str(t.source)
            if hasattr(t, 'relation'): r_str = t.relation.label if hasattr(t.relation, 'label') else str(t.relation)
            if hasattr(t, 'target'): o_str = t.target.label if hasattr(t.target, 'label') else str(t.target)
        
        # Formato de la terna
        return f"$\\bullet$ {escape_latex(s_str)} $\\xrightarrow{{\\text{{{escape_latex(r_str)}}}}}$ {escape_latex(o_str)}"

    # --- HEADER ---
    print(f"% --- Table for: {label} ---")
    # Nota: >{\raggedright\arraybackslash} es CRÍTICO. Requiere \usepackage{array} en main.tex
    print("\\begin{longtable}{|p{1.5cm}|>{\\raggedright\\arraybackslash}p{13cm}|}") 
    print(f"\\caption{{{caption}}} \\label{{{label}}} \\\\")
    print("\\hline")
    print("\\textbf{Ranking} & \\textbf{Contenido} \\\\")
    print("\\hline")
    print("\\endfirsthead")
    
    print("\\hline")
    print("\\textbf{Ranking} & \\textbf{Contenido (cont.)} \\\\")
    print("\\hline")
    print("\\endhead")

    # --- BODY ---
    for i, cmap in enumerate(retrieved_maps):
        fq = escape_latex(getattr(cmap, 'focus_question', ''))
        triples = getattr(cmap, 'triples', [])
        
        # FILA 1: La Pregunta (Lleva el número de Ranking en la Col 1)
        # IMPORTANTE: Termina con \\ (fin de fila), no \newline
        print(f"{i} & \\textbf{{Pregunta:}} {fq} \\\\") 
        
        # FILA 2: Etiqueta "Ternas" (Col 1 vacía)
        if triples:
             print(f" & \\textit{{Ternas:}} \\\\")
        
        # FILAS 3...N: Las Ternas (Una fila de tabla por cada terna)
        if not triples:
            print(f" & (Sin ternas) \\\\")
        else:
            for t in triples:
                triple_str = format_triple_content(t)
                # Col 1 vacía, Col 2 tiene la terna. Termina con \\
                print(f" & {triple_str} \\\\")

        # SEPARADOR: Línea horizontal SOLO al final del mapa completo
        print("\\hline")

    print("\\end{longtable}")
    print("\n" * 2)

In [14]:
generate_cmap_latex_atomic(
    gemini_cmap_results[1].retrieved_cmaps,
    "Contexto recuperado (Mapas Conceptuales) para Caso 2: Hemisferios del Cerebro",
    "tab:context_c2_cmap")

% --- Table for: tab:context_c2_cmap ---
\begin{longtable}{|p{1.5cm}|>{\raggedright\arraybackslash}p{13cm}|}
\caption{Contexto recuperado (Mapas Conceptuales) para Caso 2: Hemisferios del Cerebro} \label{tab:context_c2_cmap} \\
\hline
\textbf{Ranking} & \textbf{Contenido} \\
\hline
\endfirsthead
\hline
\textbf{Ranking} & \textbf{Contenido (cont.)} \\
\hline
\endhead
0 & \textbf{Pregunta:} what is the brain, what are its characteristics, and how does it develop? \\
 & \textit{Ternas:} \\
 & $\bullet$ brain $\xrightarrow{\text{is the}}$ enlarged, anterior part of the central nervous system within the skull \\
 & $\bullet$ young adult human brain $\xrightarrow{\text{weighs about}}$ 1,450 g \\
 & $\bullet$ brain outer layer $\xrightarrow{\text{contains}}$ over 10 billion nerve cells \\
 & $\bullet$ cerebral cortex $\xrightarrow{\text{contains}}$ over 10 billion nerve cells \\
 & $\bullet$ brain $\xrightarrow{\text{develops by}}$ differentiation of the embryonic neural tube \\
 & $\bullet$ 

In [23]:
print(len(gemini_cmap_results[1].retrieved_cmaps))

5
